In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [57]:
#print(files)

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [58]:
#documents

In [ ]:
# Q.1 How many lesson pages
len(documents)

72

In [8]:
# Q.2 Indexation and search
from minsearch import Index
def build_index(documents):
    index=Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

index=build_index(documents)

In [ ]:
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query=query,
    num_results=5
)

#results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

# Q2:
Filename of the first result is 
`01-agentic-rag/lessons/14-agentic-loop.md`

# Q3 RAG

In [12]:
def build_context(search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
            #lines.append('A: ' + doc['answer'])
            lines.append('')

        return '\n'.join(lines).strip()
context=build_context(results)
print(context)

01-agentic-rag/lessons/14-agentic-loop.md
content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Tools, the functions the agent can call

In [15]:

INSTRUCTIONS = """
You are a helpful teaching assistant for the LLM Zoomcamp.

Answer questions using only the provided lesson content.

If the answer cannot be found in the provided context,
reply with "I don't know."
"""

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

def build_prompt(query, search_results):
        context = build_context(search_results)
        return PROMPT_TEMPLATE.format(
            question=query, context=context
        )

prompt=build_prompt(query,results)
print(prompt)

QUESTION: How does the agentic loop keep calling the model until it stops?

CONTEXT:
01-agentic-rag/lessons/14-agentic-loop.md
content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the 

In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
from openai import OpenAI
llm_client = OpenAI()

def llm(prompt,INSTRUCTIONS=INSTRUCTIONS,):
        input_messages = [
            {'role': 'developer', 'content': INSTRUCTIONS},
            {'role': 'user', 'content': prompt}
        ]

        response = llm_client.responses.create(
            model='gpt-5.4-mini',
            input=input_messages
        )

        return response.output_text
output=llm(prompt)

In [19]:
output

'The loop keeps calling the model by checking whether the response contains any `function_call` items.\n\n- After each model call, the code runs the tool calls and appends the tool outputs to the message history.\n- It sets `has_function_calls = True` if any tool was called.\n- Then it repeats the `while True` loop.\n- The loop stops when the model returns a response with **no function calls**:\n  ```python\n  if has_function_calls == False:\n      break\n  ```\n\nSo the stop condition is simple: **no function calls this turn means the agent is done.**'

In [26]:
INSTRUCTIONS = """
You are a helpful teaching assistant for the LLM Zoomcamp.

Answer questions using only the provided lesson content.

If the answer cannot be found in the provided context,
reply with "I don't know."
"""

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,

        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):

        return self.index.search(
            query,
            num_results=num_results,
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
           
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

In [27]:
query='How does the agentic loop keep calling the model until it stops?'
assistant = RAGBase(
    index,
    llm_client=llm_client,
)

answer = assistant.rag(query)
print(answer)

Response(id='resp_0dc040bcc04c3b29006a2a815d156c819d9a476d106249ea51', created_at=1781170525.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0dc040bcc04c3b29006a2a815dcd18819d854724ee36204706', content=[ResponseOutputText(annotations=[], text='The loop keeps calling the model by checking whether the latest response contains any `function_call` items.\n\n- If there **is** a function call, the code runs the tool, appends the tool result to `messages`, and loops again.\n- If there are **no** function calls, the `has_function_calls` flag stays `False`, and the loop `break`s.\n\nSo the stop condition is: **no function calls this turn means the agent is done.**', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, comp

In [29]:
answer.usage.input_tokens

7114

input token is 7000 

# Q4

In [32]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

# Q6

In [33]:
# index chunk
index_chunk=build_index(chunks)

In [34]:
query='How does the agentic loop keep calling the model until it stops?'
assistant = RAGBase(
    index_chunk,
    llm_client=llm_client,
)

answer = assistant.rag(query)
print(answer)

Response(id='resp_063c8a61f4885d6f006a2a841af354819e8acc225717b400c6', created_at=1781171227.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_063c8a61f4885d6f006a2a841baa2c819e8ed7d557ad00579f', content=[ResponseOutputText(annotations=[], text='The loop keeps calling the model in a `while True` loop and checks each response for function calls.\n\n- If the model returns a `function_call`, the code runs the tool, adds the result to `messages`, and keeps looping.\n- If the model returns only a final `message` and no function calls, `has_function_calls` stays `False`, and the loop `break`s.\n\nSo the stopping condition is: **no function calls in that turn means the loop is done.**', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, b

In [36]:
answer.usage.input_tokens

2297

In [37]:
7114/2297

3.0970831519373094

3 fois moins

# Q6

In [45]:
def search(query: str) -> str:
    """Search for information in the course lessons."""
    results = index_chunk.search(query, num_results=5)

    return "\n\n".join(
        doc["content"] for doc in results
    )

In [47]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [49]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search for information in the course lessons.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [50]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [51]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [55]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG lesson agentic loop plain RAG"}', call_id='call_O6xZKe7FISzppsLDCO4MjVKF', name='search', type='function_call', id='fc_092d30ae2e0f74e2006a2a88c21158819c89836e72c0f0b08b', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop retrieval augmented generation difference"}', call_id='call_NuEwsrDdEHJXEfjTqXcqR0Sj', name='search', type='function_call', id='fc_092d30ae2e0f74e2006a2a88c21168819c806efd9053958942', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"plain RAG versus agentic loop co

The agent called `search` 4